# Test des fonctions possible de cvhull

In [1]:
import numpy as np
import pandas as pd
import convexhull
from convexhull import cvhull
from convexhull.fonction import f_eq

In [2]:
help(convexhull)

Help on package convexhull:

NAME
    convexhull - Calcul d'enveloppe convexe, au sens thermodynamiques, à n dimensions.

PACKAGE CONTENTS
    fonction

CLASSES
    builtins.object
        cvhull
    
    class cvhull(builtins.object)
     |  cvhull(nom, composition, energie, accuracy=4)
     |  
     |  Déterminer la stabilité des phases par calcul d'enveloppe convexe
     |  Elle prend en entrée une nombre N de phases de c éléments et caculs les phases stables et instables
     |  
     |  Paramètres:
     |  -----------
     |  nom | [str(N)]                : liste des noms des phases d'entrée
     |  composition | [[float(c)](N)] : liste des compsitions des phases
     |  energie | [flaot(N)]          : liste des énergie des phases
     |  accuracy | int (default=4)        : nombre de chiffres après la virgule utlisé pour les compositions
     |  
     |  Atributs:
     |  ---------
     |  nom | [str(N)] : Liste des nom des composés d'entrée
     |  
     |  composition | np.array

In [3]:
help(cvhull)

Help on class cvhull in module convexhull:

class cvhull(builtins.object)
 |  cvhull(nom, composition, energie, accuracy=4)
 |  
 |  Déterminer la stabilité des phases par calcul d'enveloppe convexe
 |  Elle prend en entrée une nombre N de phases de c éléments et caculs les phases stables et instables
 |  
 |  Paramètres:
 |  -----------
 |  nom | [str(N)]                : liste des noms des phases d'entrée
 |  composition | [[float(c)](N)] : liste des compsitions des phases
 |  energie | [flaot(N)]          : liste des énergie des phases
 |  accuracy | int (default=4)        : nombre de chiffres après la virgule utlisé pour les compositions
 |  
 |  Atributs:
 |  ---------
 |  nom | [str(N)] : Liste des nom des composés d'entrée
 |  
 |  composition | np.array([N,c],float) : Liste des compsitions des composés
 |  energie     | np.array([N],float)   : Liste des énergie des composés
 |  points      | np.array([N,c],float) : Liste de points dans le format quickhull 
 |                   

## Test 1 : Si on prend seulement ndim points

In [4]:
df=pd.read_csv("data_test/data.csv")
df=df[0:4]
N=len(df)
df

,nom,A,B,C,D,somme,energie
0,Fe,1,0,0,0,1,-8.19
1,Co,0,0,1,0,1,-7.11
2,Ni,0,0,0,1,1,-5.37
3,Cu,0,1,0,0,1,-3.94


In [5]:
nom=[x.replace(' ','') for x in df['nom']]
A=list(df['A'])
B=list(df['B'])
C=list(df['C'])
D=list(df['D'])
somme=list(df['somme'])



compo=[[A[i]/somme[i],B[i]/somme[i],C[i]/somme[i],D[i]/somme[i]] for i in range(N)]
Ef=list(df['energie'])
try:
    hull=cvhull(nom,compo,Ef)
except SystemExit:
    pass

Erreur : Il faut au minimum un nombre de points superieur à la dimensions des points
Ici il faut donc au minimum 5 points alors que vous en avez mit 4


## Test 2 : Si les seuls composé stable sont les purs

In [6]:
df=pd.read_csv("data_test/data-purs.csv")
N=len(df)
df

,nom,A,B,C,D,somme,energie
0,Fe,1,0,0,0,1,-8.19
1,Co,0,0,1,0,1,-7.11
2,Ni,0,0,0,1,1,-5.37
3,Cu,0,1,0,0,1,-3.94
4,CuNi2,0,1,0,1,2,-4.00


In [7]:
nom=[x.replace(' ','') for x in df['nom']]
A=list(df['A'])
B=list(df['B'])
C=list(df['C'])
D=list(df['D'])
somme=list(df['somme'])



compo=[[A[i]/somme[i],B[i]/somme[i],C[i]/somme[i],D[i]/somme[i]] for i in range(N)]
Ef=list(df['energie'])

In [8]:
hull=cvhull(nom,compo,Ef)

-----------------------------------
Seul les composés purs sont stables
-----------------------------------


In [9]:
print('Nombres de composés:',hull.npoints)
print('Dont',len(hull.stable),'stables')


Nombres de composés: 5
Dont 4 stables


In [10]:
x=[1/3,1/3,0,1/3]
for param in hull.equations:
    print(f_eq(x[:-1],param))

-5.833333333333333


In [11]:
print('Les composés stables sont les suivants :')
print('----------------------------------------')
for i in hull.stable :
    name=hull.nom[i]
    compo=hull.composition[i]
    ener=hull.energie[i]
    print('{:6} {:6.2f} {:6.2f} {:6.2f}'.format(name,ener,hull.energie_compose(name),hull.energie_hull(compo)[0]))

Les composés stables sont les suivants :
----------------------------------------
Fe      -8.19  -8.19  -8.19
Cu      -3.94  -3.94  -3.94
Co      -7.11  -7.11  -7.11
Ni      -5.37  -5.37  -5.37


## Test 3 : Exemple simple

In [12]:
df=pd.read_csv("data_test/data.csv")
N=len(df)
df

,nom,A,B,C,D,somme,energie
0,Fe,1,0,0,0,1,-8.19
1,Co,0,0,1,0,1,-7.11
2,Ni,0,0,0,1,1,-5.37
3,Cu,0,1,0,0,1,-3.94
4,CuNi,0,1,0,1,2,-4.91
5,FeCuNi,1,1,0,1,3,-6.05
6,CoNi,0,0,1,1,2,-7.00
7,CoNi2,0,0,1,1,2,-6.50


In [13]:
nom=[x.replace(' ','') for x in df['nom']]
A=list(df['A'])
B=list(df['B'])
C=list(df['C'])
D=list(df['D'])
somme=list(df['somme'])



compo=[[A[i]/somme[i],B[i]/somme[i],C[i]/somme[i],D[i]/somme[i]] for i in range(N)]
Ef=list(df['energie'])

In [14]:
hull=cvhull(nom,compo,Ef)

In [15]:
print('Nombres de composés:',hull.npoints)
print('Dont',len(hull.stable),'stables')


Nombres de composés: 8
Dont 7 stables


In [16]:
print('Les composés stables sont les suivants :')
print('----------------------------------------')
for i in hull.stable :
    name=hull.nom[i]
    compo=hull.composition[i]
    ener=hull.energie[i]
    print('{:6} {:6.2f} {:6.2f} {:6.2f}'.format(name,ener,hull.energie_compose(name),hull.energie_hull(compo)[0]))

Les composés stables sont les suivants :
----------------------------------------
Fe      -8.19  -8.19  -8.19
Co      -7.11  -7.11  -7.11
Ni      -5.37  -5.37  -5.37
Cu      -3.94  -3.94  -3.94
CuNi    -4.91  -4.91  -4.91
FeCuNi  -6.05  -6.05  -6.05
CoNi    -7.00  -7.00  -7.00


In [17]:
nom='FeCuNi'

print(hull.distance_hull(nom))

Ce composé est stable
(-8.881784197001252e-16, True)


In [18]:
nom='CuNi'

print(hull.distance_hull(nom))

Ce composé est stable
(0.0, True)


In [19]:
x=[1/3,1/3,0,1/3]
print('A la composition x')
hull.phases_stables(x)

A la composition x
Les phases stables sont :
1.0000(FeCuNi)


In [20]:
x=[1/4,1/4,1/4,1/4]
print('A la composition x')
hull.phases_stables(x)

A la composition x
Les phases stables sont :
0.5000(CoNi) + 0.2500(Cu) + 0.2500(Fe)
